Plan:
Generate segmentation predictions for the same 61 validation images using the previously trained CR-Semi-derived 2D model.

The experiment uses the saved checkpoint from epoch 231 (validation Dice: 0.43119).

No retraining is performed. BiomedParse pseudo-labels are used as weak reference annotations, not expert ground truth.

In [8]:
# Mount Google Drive and define project paths.

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

BIOMEDPARSE_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project"
)



MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

CRSEMI_CHECKPOINT = (
    BIOMEDPARSE_DIR
    / "crsemi_2d_fold0_final"
    / "checkpoint_best.pth"
)

print("Manifest exists:", MANIFEST_PATH.exists())
print("CR-Semi checkpoint exists:", CRSEMI_CHECKPOINT.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifest exists: True
CR-Semi checkpoint exists: True


In [9]:
# Mount Google Drive and define project paths.

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

VLM_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

BIOMEDPARSE_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project"
)

NNUNET_RAW_DIR = (
    BIOMEDPARSE_DIR / "nnunet_raw" / "Dataset001_Follicles"
)

MANIFEST_PATH = VLM_DIR / "image_manifest.csv"

print("VLM project:", VLM_DIR.exists())
print("BiomedParse project:", BIOMEDPARSE_DIR.exists())
print("Dataset:", NNUNET_RAW_DIR.exists())
print("Manifest:", MANIFEST_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
VLM project: True
BiomedParse project: True
Dataset: True
Manifest: True


Mount Google Drive and define project paths

In [2]:
# Load the original CR-Semi-derived 2D backbone architecture.

import sys
from pathlib import Path

MODEL_CODE_DIR = (
    BIOMEDPARSE_DIR / "crsemi_2d_inference_code"
)

assert (MODEL_CODE_DIR / "asym_backbone_2D.py").is_file(), (
    "Original model architecture file not found."
)

sys.path.insert(0, str(MODEL_CODE_DIR))

from asym_backbone_2D import asym_model_2D

print("CR-Semi-derived 2D backbone imported successfully.")

CR-Semi-derived 2D backbone imported successfully.


Load the original CR-Semi-derived 2D backbone architecture

In [3]:
# Load and validate the trained CR-Semi-derived 2D backbone.

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Initialize the original 2D backbone.
model = asym_model_2D(
    n_channels=1,
    n_classes=2
).to(device)

# Load the saved checkpoint without executing arbitrary pickle code.
checkpoint = torch.load(
    CRSEMI_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Best validation Dice:", checkpoint["best_dice"])

# The training checkpoint contains the full PointRend model.
# Extract only the backbone weights used during validation.
full_state_dict = checkpoint["model_state_dict"]

backbone_state_dict = {
    key.removeprefix("backbone."): value
    for key, value in full_state_dict.items()
    if key.startswith("backbone.")
}

assert backbone_state_dict, "No backbone weights found in checkpoint."

# Verify exact compatibility with the original architecture.
model.load_state_dict(backbone_state_dict, strict=True)

model.eval()

print("Backbone weights loaded successfully.")
print("Number of weight tensors:", len(backbone_state_dict))

Device: cuda
Checkpoint epoch: 231
Best validation Dice: 0.4311909124922563
Backbone weights loaded successfully.
Number of weight tensors: 36


Restore the previously trained backbone weights and verify their compatibility with the original architecture

In [ ]:
# Generate CR-Semi-derived 2D predictions for the 61 validation images.

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from pathlib import Path
from PIL import Image, ImageOps

MANIFEST_PATH = PROJECT_DIR / "image_manifest.csv"

OUTPUT_DIR = (
    PROJECT_DIR / "predictions" / "crsemi_2d_validation"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the same validation cases used for Gemini and nnU-Net.
manifest = pd.read_csv(MANIFEST_PATH)

validation = manifest.loc[
    manifest["split"].astype(str).str.strip().str.lower()
    .isin(["val", "validation"])
].copy()

assert len(validation) == 61
assert validation["case_id"].is_unique


def preprocess_image(image, target_size=(1024, 768)):
    """Resize and pad exactly as in the original 2D dataset."""

    target_h, target_w = target_size
    original_w, original_h = image.size

    scale = min(
        target_w / original_w,
        target_h / original_h
    )

    new_w = int(round(original_w * scale))
    new_h = int(round(original_h * scale))

    resized = image.resize(
        (new_w, new_h),
        resample=Image.Resampling.BILINEAR
    )

    pad_left = (target_w - new_w) // 2
    pad_top = (target_h - new_h) // 2

    padded = ImageOps.expand(
        resized,
        border=(
            pad_left,
            pad_top,
            target_w - new_w - pad_left,
            target_h - new_h - pad_top
        ),
        fill=0
    )

    array = np.asarray(padded, dtype=np.float32) / 255.0

    tensor = torch.from_numpy(array)[None, None]

    metadata = {
        "original_size": (original_w, original_h),
        "resized_size": (new_w, new_h),
        "pad_left": pad_left,
        "pad_top": pad_top
    }

    return tensor, metadata


model.eval()

saved_cases = []

for row in validation.itertuples(index=False):

    output_path = OUTPUT_DIR / f"{row.case_id}.png"

    # Preserve any existing predictions.
    if output_path.exists():
        saved_cases.append(row.case_id)
        continue

    image = Image.open(row.image_path).convert("L")

    input_tensor, metadata = preprocess_image(image)
    input_tensor = input_tensor.to(device)

    with torch.inference_mode():

        # Use the coarse output, matching the training validation procedure.
        result = model(input_tensor)

        coarse_logits = result["coarse"]

        upsampled_logits = F.interpolate(
            coarse_logits,
            size=input_tensor.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        predicted_mask = (
            upsampled_logits.argmax(dim=1)[0]
            .cpu()
            .numpy()
            .astype(np.uint8)
        )

    # Remove padding.
    new_w, new_h = metadata["resized_size"]
    left = metadata["pad_left"]
    top = metadata["pad_top"]

    cropped_mask = predicted_mask[
        top:top + new_h,
        left:left + new_w
    ]

    # Restore original image dimensions.
    original_w, original_h = metadata["original_size"]

    restored_mask = Image.fromarray(cropped_mask).resize(
        (original_w, original_h),
        resample=Image.Resampling.NEAREST
    )

    restored_mask.save(output_path)

    saved_cases.append(row.case_id)

print("Processed or already saved:", len(saved_cases))
print("Output directory:", OUTPUT_DIR)

Processed or already saved: 61
Output directory: /content/drive/MyDrive/ovarian_ultrasound_vlm/predictions/crsemi_2d_validation


Apply the original training preprocessing, run coarse segmentation inference, and restore predicted masks to the original ultrasound image dimensions

In [ ]:
# Validate the saved CR-Semi-derived 2D masks.

from PIL import Image
import numpy as np

expected_names = {
    f"{case_id}.png"
    for case_id in validation["case_id"]
}

actual_names = {
    path.name
    for path in OUTPUT_DIR.glob("*.png")
}

assert actual_names == expected_names, (
    f"Missing: {expected_names - actual_names}; "
    f"Unexpected: {actual_names - expected_names}"
)

for row in validation.itertuples(index=False):

    mask = np.asarray(
        Image.open(OUTPUT_DIR / f"{row.case_id}.png")
    )

    original = Image.open(row.image_path)

    assert mask.shape == (original.height, original.width)
    assert set(np.unique(mask)).issubset({0, 1})

print("Validated masks:", len(actual_names))
print("All 61 CR-Semi predictions are valid.")

Validated masks: 61
All 61 CR-Semi predictions are valid.


Validate the saved CR-Semi-derived 2D masks

In [ ]:
# Generate CR-Semi-derived 2D qualitative review figures.

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from skimage import measure

CRSEMI_REVIEW_DIR = (
    PROJECT_DIR / "figures"
    / "crsemi_2d_qualitative"
    / "qualitative_review"
)

CRSEMI_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

assert CRSEMI_REVIEW_DIR.resolve() != (
    PROJECT_DIR / "figures" / "semantic_evaluation" / "qualitative_review"
).resolve()

assert CRSEMI_REVIEW_DIR.resolve() != (
    PROJECT_DIR / "figures" / "biomedparse_qualitative"
    / "qualitative_review_comparison"
).resolve()

for row in validation.itertuples(index=False):
    case_id = row.case_id

    image = np.asarray(
        Image.open(row.image_path).convert("RGB")
    )

    mask = np.asarray(
        Image.open(OUTPUT_DIR / f"{case_id}.png")
    )

    assert mask.shape == image.shape[:2], case_id

    binary_mask = mask == 1
    region_count = measure.label(
        binary_mask, connectivity=2
    ).max()

    overlay = np.zeros((*binary_mask.shape, 4), dtype=np.float32)
    overlay[binary_mask, :3] = [0.0, 0.5, 1.0]
    overlay[binary_mask, 3] = 0.30

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(image)
    axes[0].set_title("Original Ultrasound", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(image)
    axes[1].imshow(overlay)
    axes[1].set_title("CR-Semi-derived 2D Prediction", fontsize=14)
    axes[1].axis("off")

    fig.suptitle(
        f"{case_id} | Predicted regions: {region_count}",
        fontsize=15
    )

    fig.subplots_adjust(
        left=0.02, right=0.98,
        bottom=0.04, top=0.90,
        wspace=0.06
    )

    output_path = CRSEMI_REVIEW_DIR / f"{case_id}_review.png"

    if not output_path.exists():
        fig.savefig(output_path, dpi=180, bbox_inches="tight")

    plt.close(fig)

review_files = list(CRSEMI_REVIEW_DIR.glob("*_review.png"))

assert len(review_files) == 61

print("CR-Semi review figures:", len(review_files))
print("Saved to:", CRSEMI_REVIEW_DIR)

CR-Semi review figures: 61
Saved to: /content/drive/MyDrive/ovarian_ultrasound_vlm/figures/crsemi_2d_qualitative/qualitative_review


Create side-by-side visualizations of the original ultrasound images and CR-Semi-derived 2D segmentation predictions

In [ ]:
# Create a CR-Semi review table using the BiomedParse review workflow.

import pandas as pd
import numpy as np
from PIL import Image
from skimage import measure
from pathlib import Path
from datetime import datetime
import shutil

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.formatting.rule import CellIsRule
from openpyxl.utils import get_column_letter

# Define review output paths.
REVIEW_DIR = (
    PROJECT_DIR / "evaluation" / "crsemi_2d_qualitative"
)
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_PATH = REVIEW_DIR / "crsemi_2d_manual_review.xlsx"

review_records = []

for _, row in validation.iterrows():

    case_id = row["case_id"]

    # Load the CR-Semi predicted segmentation mask.
    mask_path = OUTPUT_DIR / f"{case_id}.png"
    mask = np.array(Image.open(mask_path))

    assert mask.ndim == 2, f"Expected a 2D mask: {case_id}"

    # Identify connected components using 8-connectivity.
    binary_mask = mask == 1
    labeled_mask = measure.label(binary_mask, connectivity=2)
    regions = measure.regionprops(labeled_mask)

    review_records.append({
        "case_id": case_id,
        "category": row["category"],
        "crsemi_count": len(regions),
        "review_figure": str(
            CRSEMI_REVIEW_DIR / f"{case_id}_review.png"
        ),
        "overall_visual_assessment": "",
        "reviewer_notes": ""
    })

review_template = pd.DataFrame(review_records)

# Validate the review table.
assert len(review_template) == 61
assert review_template["case_id"].is_unique

assert review_template["review_figure"].apply(
    lambda path: Path(path).is_file()
).all()

# Preserve existing manual assessments and reviewer notes.
if REVIEW_PATH.exists():

    existing_review = pd.read_excel(
        REVIEW_PATH,
        sheet_name="Manual Review",
        keep_default_na=False
    )

    assert existing_review["case_id"].is_unique

    existing_review = existing_review.set_index("case_id")

    for column in [
        "overall_visual_assessment",
        "reviewer_notes"
    ]:
        if column in existing_review.columns:
            review_template[column] = (
                review_template["case_id"]
                .map(existing_review[column])
                .fillna("")
            )

    # Back up the original workbook before updating it.
    backup_path = REVIEW_PATH.with_name(
        f"{REVIEW_PATH.stem}_backup_"
        f"{datetime.now():%Y%m%d_%H%M%S_%f}.xlsx"
    )
    shutil.copy2(REVIEW_PATH, backup_path)

    print("Existing manual reviews preserved.")
    print("Backup:", backup_path)

# Create a clean, consistently structured Excel workbook.
wb = Workbook()
ws = wb.active
ws.title = "Manual Review"

# Write column headers.
ws.append(review_template.columns.tolist())

# Write review records.
for record in review_template.itertuples(index=False, name=None):
    ws.append(list(record))

# Style the header.
for cell in ws[1]:
    cell.fill = PatternFill(
        "solid", fgColor="183153"
    )
    cell.font = Font(
        name="Calibri",
        size=11,
        bold=True,
        color="FFFFFF"
    )
    cell.alignment = Alignment(
        vertical="center",
        wrap_text=True
    )

ws.row_dimensions[1].height = 34
ws.freeze_panes = "C2"
ws.auto_filter.ref = ws.dimensions
ws.sheet_view.showGridLines = False

# Set column widths.
column_widths = {
    "case_id": 23,
    "category": 16,
    "crsemi_count": 18,
    "review_figure": 65,
    "overall_visual_assessment": 30,
    "reviewer_notes": 70
}

for col_idx, column_name in enumerate(review_template.columns, start=1):
    letter = get_column_letter(col_idx)
    ws.column_dimensions[letter].width = column_widths[column_name]

# Find the editable review columns.
assessment_col = (
    review_template.columns.get_loc("overall_visual_assessment") + 1
)
notes_col = (
    review_template.columns.get_loc("reviewer_notes") + 1
)

assessment_letter = get_column_letter(assessment_col)
assessment_range = f"{assessment_letter}2:{assessment_letter}{ws.max_row}"

# Add a dropdown for manual assessment.
dropdown = DataValidation(
    type="list",
    formula1='"Good,Partial,Poor,Unclear"',
    allow_blank=True
)

dropdown.error = "Select Good, Partial, Poor, or Unclear."
dropdown.errorTitle = "Invalid assessment"
dropdown.showErrorMessage = True
dropdown.errorStyle = "stop"

ws.add_data_validation(dropdown)
dropdown.add(assessment_range)

# Style data cells.
for row_idx in range(2, ws.max_row + 1):

    ws.row_dimensions[row_idx].height = 34

    for cell in ws[row_idx]:
        cell.font = Font(name="Calibri", size=11)
        cell.alignment = Alignment(
            vertical="center",
            wrap_text=True
        )

    # Imported model data: green text.
    ws.cell(row_idx, 3).font = Font(
        name="Calibri", size=11, color="008000"
    )

    # Editable assessment field.
    assessment_cell = ws.cell(row_idx, assessment_col)
    assessment_cell.fill = PatternFill(
        "solid", fgColor="FFF2CC"
    )
    assessment_cell.font = Font(
        name="Calibri", size=11, color="1647A6"
    )

    # Editable reviewer notes.
    notes_cell = ws.cell(row_idx, notes_col)
    notes_cell.font = Font(
        name="Calibri", size=11, color="1647A6"
    )

    # Make the figure path clickable.
    figure_cell = ws.cell(row_idx, 4)
    figure_cell.hyperlink = figure_cell.value
    figure_cell.font = Font(
        name="Calibri",
        size=11,
        color="0563C1",
        underline="single"
    )

# Automatically color assessment results.
assessment_colors = {
    "Good": "D9EAD3",
    "Partial": "FFF2CC",
    "Poor": "F4CCCC",
    "Unclear": "E7E6E6"
}

for assessment, color in assessment_colors.items():
    ws.conditional_formatting.add(
        assessment_range,
        CellIsRule(
            operator="equal",
            formula=[f'"{assessment}"'],
            fill=PatternFill("solid", fgColor=color)
        )
    )

# Save the finished workbook.
wb.save(REVIEW_PATH)

print("CR-Semi review table saved:", REVIEW_PATH)
print("Images for review:", len(review_template))
print("Columns:", review_template.columns.tolist())

display(review_template.head())

Existing manual reviews preserved.
Backup: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/crsemi_2d_qualitative/crsemi_2d_manual_review_backup_20261102_192934_928109.xlsx
CR-Semi review table saved: /content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/crsemi_2d_qualitative/crsemi_2d_manual_review.xlsx
Images for review: 61
Columns: ['case_id', 'category', 'crsemi_count', 'review_figure', 'overall_visual_assessment', 'reviewer_notes']


,case_id,category,crsemi_count,review_figure,overall_visual_assessment,reviewer_notes
0,dominant_0011,dominant,8,/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
1,dominant_0012,dominant,2,/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
2,dominant_0017,dominant,3,/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
3,dominant_0019,dominant,3,/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,
4,dominant_0021,dominant,3,/content/drive/MyDrive/ovarian_ultrasound_vlm/...,,


Create a structured spreadsheet for manual visual assessment of CR-Semi-derived 2D predictions.

The assessment uses four categories: Good, Partial, Poor, and Unclear

In [6]:
# Summarize the completed CR-Semi qualitative review.

import pandas as pd

REVIEW_DIR = (
    PROJECT_DIR / "evaluation" / "crsemi_2d_qualitative"
)
REVIEW_PATH = REVIEW_DIR / "crsemi_2d_manual_review.xlsx"

review_df = pd.read_excel(
    REVIEW_PATH,
    sheet_name="Manual Review",
    engine="openpyxl",
    keep_default_na=False
)

assert len(review_df) == 61
assert review_df["case_id"].is_unique

valid_assessments = ["Good", "Partial", "Poor", "Unclear"]

review_df["overall_visual_assessment"] = (
    review_df["overall_visual_assessment"]
    .astype(str)
    .str.strip()
)

invalid = review_df.loc[
    ~review_df["overall_visual_assessment"].isin(valid_assessments),
    ["case_id", "overall_visual_assessment"]
]

assert invalid.empty, (
    f"Missing or invalid assessments:\n{invalid.to_string(index=False)}"
)

summary = (
    review_df["overall_visual_assessment"]
    .value_counts()
    .reindex(valid_assessments, fill_value=0)
    .rename_axis("assessment")
    .reset_index(name="count")
)

summary["percentage"] = (
    summary["count"] / len(review_df) * 100
).round(2)

# Save the full manual review and aggregate results separately.
EVALUATION_DIR = REVIEW_PATH.parent

FINAL_REVIEW_CSV = (
    EVALUATION_DIR / "crsemi_2d_manual_review_final.csv"
)

SUMMARY_CSV = (
    EVALUATION_DIR / "crsemi_2d_qualitative_summary_final.csv"
)

review_df.to_csv(FINAL_REVIEW_CSV, index=False)
summary.to_csv(SUMMARY_CSV, index=False)

print("Reviewed images:", len(review_df))
print("\nAssessment summary:")
display(summary)

print("\nSaved files:")
print(FINAL_REVIEW_CSV)
print(SUMMARY_CSV)

Reviewed images: 61

Assessment summary:


,assessment,count,percentage
0,Good,2,3.28
1,Partial,51,83.61
2,Poor,6,9.84
3,Unclear,2,3.28



Saved files:
/content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/crsemi_2d_qualitative/crsemi_2d_manual_review_final.csv
/content/drive/MyDrive/ovarian_ultrasound_vlm/evaluation/crsemi_2d_qualitative/crsemi_2d_qualitative_summary_final.csv


Aggregate manual visual assessments across the 61 validation images and save the results for cross-model comparison

In [11]:
# Generate nnU-Net 2D qualitative review figures.

import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
from PIL import Image
from skimage import measure

NNUNET_PRED_DIR = (
    VLM_DIR / "predictions" / "nnunet_2d_validation"
)

NNUNET_REVIEW_DIR = (
    VLM_DIR / "figures"
    / "nnunet_2d_qualitative"
    / "qualitative_review"
)

NNUNET_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)

validation = manifest.loc[
    manifest["split"].astype(str).str.strip().str.lower()
    .isin(["val", "validation"])
].copy()

assert len(validation) == 61

for row in validation.itertuples(index=False):

    case_id = row.case_id

    image = np.asarray(
        Image.open(row.image_path).convert("RGB")
    )

    mask = np.asarray(
        Image.open(NNUNET_PRED_DIR / f"{case_id}.png")
    )

    assert mask.shape == image.shape[:2], case_id

    binary_mask = mask == 1

    region_count = int(
        measure.label(binary_mask, connectivity=2).max()
    )

    overlay = np.zeros(
        (*binary_mask.shape, 4),
        dtype=np.float32
    )

    overlay[binary_mask, :3] = [0.0, 0.5, 1.0]
    overlay[binary_mask, 3] = 0.30

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(image)
    axes[0].set_title("Original Ultrasound", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(image)
    axes[1].imshow(overlay)
    axes[1].set_title("nnU-Net 2D Prediction", fontsize=14)
    axes[1].axis("off")

    fig.suptitle(
        f"{case_id} | Predicted regions: {region_count}",
        fontsize=15
    )

    fig.subplots_adjust(
        left=0.02,
        right=0.98,
        bottom=0.04,
        top=0.90,
        wspace=0.06
    )

    output_path = (
        NNUNET_REVIEW_DIR / f"{case_id}_review.png"
    )

    if not output_path.exists():
        fig.savefig(
            output_path,
            dpi=180,
            bbox_inches="tight"
        )

    plt.close(fig)

review_files = list(
    NNUNET_REVIEW_DIR.glob("*_review.png")
)

assert len(review_files) == 61

print("nnU-Net review figures:", len(review_files))
print("Saved to:", NNUNET_REVIEW_DIR)

NameError: name 'manifest' is not defined

Generate side-by-side visualizations of the original ultrasound images and nnU-Net 2D segmentation predictions for manual assessment